# Create Modelfile
This section creates a reusable model definition so citizen developers and DevOps engineers get consistent, production-minded responses.

**Key Definitions:**
- `Modelfile`: Instruction file that tells Ollama how to build and run a custom model.
- `FROM`: Base model to inherit from.
- `PARAMETER`: Runtime tuning controls for generation behavior.
- `SYSTEM`: High-priority behavioral instructions for model responses.

**Purpose:**
Define a custom NetOps/Telecom expert model with specialized knowledge in IP networking, telecommunications, and software engineering. The Modelfile combines the base llama3.2-vision model with carefully tuned parameters (low temperature for consistency, large context window) and a comprehensive system prompt that shapes the model's personality, expertise domain, and response format. This ensures all responses are accurate, production-minded, and include actionable guidance with code/config snippets, verification steps, and pitfall warnings.

In [24]:
import os

modelfile_text = '''

FROM llama3.2-vision:latest
PARAMETER top_k 90
PARAMETER top_p 1
PARAMETER num_ctx 10000
PARAMETER temperature 0.1
PARAMETER repeat_last_n 70
PARAMETER seed 0

SYSTEM """
You are NetOps/Telecom Principal Engineer Copilot.

Scope & Expertise:
- Deep expertise in IP networking and Telecommunications: routing/switching, MPLS, BGP/OSPF/IS-IS, EVPN/VXLAN, QoS, multicast, transport/optical concepts, SIP/IMS, RTP/SRTP, VoIP, SBCs, call flows, dial plans, numbering, and troubleshooting.
- Strong software engineering skills: Python, Java (Spring), JavaScript/TypeScript, APIs, automation, testing, and CI/CD.

Behavior:
- Be accurate, pragmatic, and production-minded. Prefer actionable guidance over theory.
- Keep responses thorough but not verbose: default to concise explanations, then provide bullets/steps/examples only as needed.
- If you must assume something, say so briefly and choose the safest reasonable assumption.
- Ask at most 1–3 clarifying questions only when required to avoid a wrong answer; otherwise proceed with best-effort and note what to verify.

How to answer:
1) Start with the direct answer or recommended approach.
2) Provide the key reasoning in short bullets.
3) Provide concrete commands, config snippets, code, or checklists when useful.
4) Include verification steps and common pitfalls.
5) When presenting code/config, keep it minimal, correct, and runnable. Prefer idiomatic style and clear naming.

Vision inputs (images/screenshots/diagrams):
- If an image is provided, describe what you observe, extract relevant details (labels, alarms, counters, logs), and map them to likely root causes.
- When uncertain, list the top plausible interpretations and what evidence would confirm each.
"""
'''

with open ('Modelfile', 'w', encoding='utf-8') as f:
    f.write(modelfile_text)
print("Modelfile created at.", os.path.abspath('Modelfile'))

import subprocess, shlex
def run_command(command):
    print('$', command)
    process = subprocess.run(shlex.split(command), check=True, text=True, capture_output=True)
    print(process.stdout)
    print(process.stderr)
    return process

Modelfile created at. /Users/AC05735/DevOps/1a_DevProjects_local/Code/Ollama_Notebooks/SpringSummit26/Notebooks_prompting/Modelfile


## Build Local Model
This step builds the custom model from the Modelfile and verifies it is available for local use.

**Key Definitions:**
- `ollama create`: Builds a model artifact from a Modelfile.
- `MODEL_NAME`: Reusable local model tag used in later API calls.
- `subprocess.run`: Python API that executes shell commands from code.
- `capture_output=True`: Captures stdout and stderr for debugging and validation.

**Purpose:**
Execute the Ollama model creation process programmatically using Python subprocess management. This automation is essential for reproducible deployments and CI/CD pipelines. The function captures all output for troubleshooting, verifies the build succeeded, and confirms the model appears in the local registry. Error handling ensures that build failures are properly surfaced with diagnostic information.

In [25]:
import subprocess, shlex, os, textwrap

MODEL_NAME = "netops-llama3.2-vision"
MODEL_FILE = "Modelfile"

def run(cmd):
    print('$', cmd)
    process = subprocess.run(shlex.split(cmd), check=True, text=True, capture_output=True)
    if process.stdout.strip():
        print(process.stdout)
    if process.stderr.strip():
        print(process.stderr)
        if process.returncode != 0:
            raise subprocess.CalledProcessError(process.returncode, cmd, output=process.stdout, stderr=process.stderr)
    return process

print("Working directory:", os.getcwd())
run("ls -la")
run(f"ollama create {MODEL_NAME} -f {MODEL_FILE}")
run(textwrap.fill(f"ollama list", width=80))

Working directory: /Users/AC05735/DevOps/1a_DevProjects_local/Code/Ollama_Notebooks/SpringSummit26/Notebooks_prompting
$ ls -la
total 136
drwxr-xr-x@  4 AC05735  staff    128 Mar  8 23:12 .
drwxr-xr-x  15 AC05735  staff    480 Mar  8 19:18 ..
-rw-r--r--@  1 AC05735  staff   1731 Mar  8 23:23 Modelfile
-rw-r--r--@  1 AC05735  staff  63267 Mar  8 23:17 promptEngineering_example.ipynb

$ ollama create netops-llama3.2-vision -f Modelfile
gathering model components 
using existing layer sha256:9999d473417a8e179d993498195be5f42cab963acc75f4a6b15d981e8b68abed 
using existing layer sha256:715415638c9c4c0cb2b78783da041b97bd1205f8b9f9494bd7e5a850cb443602 
using existing layer sha256:d3ed60b917572dd5aa69bf5ff7825f2db00d65e73dc07a43fbc31c60eb31449e 
using existing layer sha256:fcc5a6bec9daf9b561a68827b67ab6088e1dba9d1fa2a50d7bbcc8384e0a265d 
using existing layer sha256:e7ef3abe2ecf05a742fd1f8a74f3d57a35560ed1e3f60888de9ef9a30f7f8fb9 
using existing layer sha256:f3db3c367f757890443d8f220aa4d940754f

CompletedProcess(args=['ollama', 'list'], returncode=0, stdout='NAME                             ID              SIZE      MODIFIED               \nnetops-llama3.2-vision:latest    bc5a436cf585    7.8 GB    Less than a second ago    \ngemma:7b                         a72c7f4d0a15    5.0 GB    21 minutes ago            \nembeddinggemma:latest            85462619ee72    621 MB    10 days ago               \nllama3.2-vision:latest           6f2f9757ae97    7.8 GB    12 days ago               \nllama3.1:8b                      46e0c10c039e    4.9 GB    6 months ago              \ngpt-oss:20b                      aa4295ac10c3    13 GB     6 months ago              \n', stderr='')

# Check Environment
This quick check confirms your Python runtime and platform so setup issues are easier to troubleshoot.

**Key Definitions:**
- `Runtime`: Active Python interpreter executing the notebook.
- `Platform`: OS and architecture details that can affect behavior.
- `Dependency drift`: Mismatch between expected and actual environment versions.

**Purpose:**
Validate the execution environment before running model operations. This diagnostic step catches common issues like Python version incompatibilities, OS-specific behavior differences (macOS vs Linux vs Windows), and architecture mismatches (ARM vs x86). Having this information upfront accelerates troubleshooting when model operations fail due to environment-specific issues rather than code errors.

In [26]:
import sys, platform, json, textwrap, os, time
print("Python:", sys.version)
print(textwrap.fill(f"Platform: {platform.platform()}", width=80))

Python: 3.12.10 (main, Sep 15 2025, 14:26:18) [Clang 17.0.0 (clang-1700.0.13.5)]
Platform: macOS-15.7.4-arm64-arm-64bit


# Ensure that Ollama is running
This verifies the Ollama service is reachable before sending model requests.

**Key Definitions:**
- `localhost:11434`: Default local Ollama API endpoint.
- `Health check`: Small request used to confirm service availability.
- `URLError`: Network-level failure (service down, unreachable host, etc.).
- `timeout`: Maximum wait time before treating a request as failed.

**Purpose:**
Perform a pre-flight health check to ensure the Ollama service is running and accepting connections. This prevents cryptic timeout errors later in the workflow by failing fast with a clear diagnostic message. Essential for automated workflows where Ollama might not be started, or when running on remote hosts where network connectivity could be an issue. The timeout prevents indefinite hangs if the service is unresponsive.

In [27]:
import urllib.request, urllib.error

def ollama_up(host="http://localhost:11434"):
    try:
        with urllib.request.urlopen(host, timeout=5) as response:
            return True, response.status
    except urllib.error.URLError as e:
        return False, str(e)

ok, info = ollama_up()
print("Ollama is up and running.", ok, "|", info)


Ollama is up and running. True | 200


# List the local models
This lists locally available models so you can confirm the expected model name before inference.

**Key Definitions:**
- `Tag`: User-facing model identifier such as `name:latest`.
- `Digest`: Content hash that uniquely identifies model bytes.
- `Quantization`: Compression level trading quality for memory and speed.
- `Parameter size`: Approximate model scale (for example, `8B`, `20B`).

**Purpose:**
Query the Ollama service to enumerate all locally available models with their metadata. This verification step confirms that the custom model was built successfully and is ready for inference. The model list helps prevent "model not found" errors by showing exact tag names, and provides visibility into model versions through digests. Useful for debugging when multiple versions of a model exist or when verifying model updates have been applied.

In [28]:
import urllib.request, json

HOST = "http://localhost:11434"

def normalize_host(host: str) -> str:
    return host.rstrip("/")

def ollama_tags(host=HOST):
    host = normalize_host(host)
    request_url = urllib.request.Request(f"{host}/api/tags", method="GET")
    with urllib.request.urlopen(request_url) as response:
        return json.loads(response.read().decode("utf-8"))

tags = ollama_tags()
print("Models in Ollama:", len(tags.get("models", [])))
print(json.dumps(tags, indent=2))


Models in Ollama: 6
{
  "models": [
    {
      "name": "netops-llama3.2-vision:latest",
      "model": "netops-llama3.2-vision:latest",
      "modified_at": "2026-03-08T23:26:04.253686278-06:00",
      "size": 7816590868,
      "digest": "bc5a436cf585f6fcc041b141c9f287cc2d7f5a8e548f72dc84db01a3f064ad3a",
      "details": {
        "parent_model": "",
        "format": "gguf",
        "family": "mllama",
        "families": [
          "mllama"
        ],
        "parameter_size": "10.7B",
        "quantization_level": "Q4_K_M"
      }
    },
    {
      "name": "gemma:7b",
      "model": "gemma:7b",
      "modified_at": "2026-03-08T23:04:23.384924394-06:00",
      "size": 5011853225,
      "digest": "a72c7f4d0a15522df81486d13ce432c79e191bda2558d024fbad4362c4f7cbf8",
      "details": {
        "parent_model": "",
        "format": "gguf",
        "family": "gemma",
        "families": [
          "gemma"
        ],
        "parameter_size": "9B",
        "quantization_level": "Q4_0"
  

## Send Text Chat Request
This section demonstrates a structured text chat call to validate model behavior and API connectivity.

**Key Definitions:**
- `messages`: Ordered conversation list sent to `/api/chat`.
- `role`: Speaker type (`system`, `user`, or `assistant`).
- `payload`: JSON body posted to the model API.
- `stream=False`: Returns one complete response instead of token streaming.
- `options`: Optional generation controls (for example `num_predict`, `temperature`).

**Purpose:**
Implement a flexible chat interface that supports both simple string queries and multi-turn conversations with system prompts. The function demonstrates how to structure messages for context-aware responses, particularly valuable for technical scenarios like network configuration where system-level instructions ensure consistent output format. The multi-turn example shows how to maintain conversation history, enabling the model to build on previous responses for more sophisticated interactions.

In [ ]:
import urllib.request, urllib.error, json

HOST = "http://localhost:11434"
MODEL_NAME = "netops-llama3.2-vision"

def normalize_host(host: str) -> str:
    return host.rstrip("/")

def chat(messages, model=MODEL_NAME, host=HOST, stream=False, options=None):
    """Send messages to Ollama chat API. Messages can be a string or list of message dicts."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    payload = {
        "model": model,
        "messages": messages,
        "stream": stream,
    }
    if options:
        payload["options"] = options

    host = normalize_host(host)
    request_data = json.dumps(payload).encode("utf-8")
    request_url = urllib.request.Request(
        f"{host}/api/chat",
        data=request_data,
        method="POST",
        headers={"Content-Type": "application/json"},
    )

    try:
        with urllib.request.urlopen(request_url) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="ignore") if hasattr(e, "read") else ""
        raise RuntimeError(f"Ollama chat failed: HTTP {e.code} {e.reason}. Body: {body}") from e

# Example: Multi-turn conversation with BGP configuration
response = chat(
    [
        # {"role": "system", "content": "You are a network engineer. Always use Loopback0 for BGP router-id and enable log-neighbor-changes."},
        # {"role": "user", "content": "Configure BGP for AS 65001 with neighbor 192.168.1.2 in AS 65002"},
        # {"role": "assistant", "content": "interface Loopback0\n description BGP_ROUTER_ID\n ip address 10.1.1.1 255.255.255.255\n!\nrouter bgp 65001\n bgp router-id 10.1.1.1\n bgp log-neighbor-changes\n neighbor 192.168.1.2 remote-as 65002\n neighbor 192.168.1.2 activate"},
    ]
)

print("Response:", response["message"]["content"])

Response: **BGP Configuration for AS 65001**

To configure BGP for AS 65001 with neighbor 192.168.1.2 in AS 65002, you can use the following configuration:

```bash
router bgp 65001
  bgp log-neighbor-changes
  neighbor 192.168.1.2 remote-as 65002
  neighbor 192.168.1.2 update-source Loopback0
  neighbor 192.168.1.2 password <password>
  neighbor 192.168.1.2 send-community
  neighbor 192.168.1.2 route-reflector-client
  network <network>
```

**Key Reasoning:**

* `router bgp 65001`: Specifies the BGP process and AS number.
* `neighbor 192.168.1.2 remote-as 65002`: Specifies the neighbor IP address, AS number, and establishes the BGP session.
* `update-source Loopback0`: Specifies the source IP address for BGP updates.
* `password <password>`: Specifies the password for BGP authentication (replace `<password>` with your actual password).
* `send-community`: Enables the sending of community attributes.
* `route-reflector-client`: Enables the neighbor as a route-reflector client.
* `netw

## Interpret Logs or Pasted Data
This section defines a helper that converts operational text data into concise, actionable JSON insights.

**Key Definitions:**
- `Prompt template`: Reusable structure used to build user input.
- `Context`: Extra situational details that improve model relevance.
- `Structured output`: Predictable format designed for automation workflows.
- `Actionable insight`: Recommendation that can be executed by an operator or pipeline.
- `file_path`: Optional single file path when data is not pasted directly.
- `file_paths`: Optional list of up to two file paths for side-by-side analysis.
- `compare`: Enables direct comparison guidance when two files are supplied.
- `max_mb`: Maximum file size read before truncation is applied.
- `max_lines`: Maximum line count sent to the model from file input.
- `truncation notice`: Message prepended when content is automatically shortened.

**Purpose:**
Create a production-grade log analysis tool that handles multiple input modes (pasted text, single file, or two-file comparison) with built-in safety guardrails. The truncation logic prevents context window overflow on large files while informing the model that data was shortened. The comparison mode enables operational use cases like configuration drift detection or change analysis. Structured output with summary, key details, root causes, next steps, and uncertainties makes responses immediately actionable for automation workflows or incident response playbooks.

In [31]:
import json, urllib.request, urllib.error, textwrap, pathlib

HOST = "http://localhost:11434"
MODEL_NAME = "netops-llama3.2-vision"

INTERPRET_INSTRUCTIONS = """
Interpret the pasted data. Output in a structured JSON format with key insights, likely root causes, and recommended next steps.
1) Summary of what the data shows (2-4 bullets)
2) Key details that support the summary (bullets, logs, counters, labels)
3) Likely root causes (bullets, logs, counters, labels)
4) Recommended next steps (bullets, actions, mitigations)
5) Any uncertainties or assumptions (bullets)
Keep the output concise and focused on actionable insights.
"""

def normalize_host(host: str) -> str:
    return host.rstrip("/")

def _truncate_text_by_lines(text: str, max_lines: int) -> tuple[str, bool, int]:
    lines = text.splitlines()
    total = len(lines)
    if total <= max_lines:
        return text, False, total
    return "\n".join(lines[:max_lines]), True, total

def _read_file_with_guardrails(file_path: str, max_mb: float, max_lines: int) -> str:
    path = pathlib.Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")

    max_bytes = int(max_mb * 1024 * 1024)
    data_bytes = path.read_bytes()
    original_size = len(data_bytes)
    truncated_by_size = original_size > max_bytes
    if truncated_by_size:
        data_bytes = data_bytes[:max_bytes]

    text = data_bytes.decode("utf-8", errors="ignore")
    text, truncated_by_lines, original_line_count = _truncate_text_by_lines(text, max_lines)

    if truncated_by_size or truncated_by_lines:
        notices = ["[TRUNCATION NOTICE]"]
        if truncated_by_size:
            notices.append(f"- File size exceeded limit ({original_size} bytes > {max_bytes} bytes).")
        if truncated_by_lines:
            notices.append(f"- File line count exceeded limit ({original_line_count} lines > {max_lines} lines).")
        notices.append("- Only the beginning portion of the file was analyzed.")
        return "\n".join(notices) + "\n\n" + text

    return text

def _load_input_text(
    pasted_data: str | None = None,
    file_path: str | None = None,
    file_paths: list[str] | None = None,
    max_mb: float = 1.0,
    max_lines: int = 4000,
) -> str:
    if pasted_data and (file_path or file_paths):
        raise ValueError("Provide pasted_data or file input, not both.")

    if file_paths and file_path:
        raise ValueError("Use either file_path or file_paths, not both.")

    if file_paths is not None:
        if len(file_paths) == 0 or len(file_paths) > 2:
            raise ValueError("file_paths must contain 1 or 2 paths.")
        blocks = []
        for idx, fp in enumerate(file_paths, start=1):
            file_text = _read_file_with_guardrails(fp, max_mb=max_mb, max_lines=max_lines)
            blocks.append(f"[FILE {idx}: {fp}]\n{file_text}")
        return "\n\n".join(blocks)

    if file_path:
        return _read_file_with_guardrails(file_path, max_mb=max_mb, max_lines=max_lines)

    if pasted_data:
        return pasted_data

    raise ValueError("Provide pasted_data, file_path, or file_paths.")

def interpret_data(
    pasted_data: str | None = None,
    context: str = "",
    file_path: str | None = None,
    file_paths: list[str] | None = None,
    compare: bool = False,
    host: str = HOST,
    timeout_seconds: int = 420,
    max_mb: float = 1.0,
    max_lines: int = 1500,
) -> str:
    if compare and (not file_paths or len(file_paths) != 2):
        raise ValueError("compare=True requires exactly two paths in file_paths.")

    data_text = _load_input_text(
        pasted_data=pasted_data,
        file_path=file_path,
        file_paths=file_paths,
        max_mb=max_mb,
        max_lines=max_lines,
    )

    compare_instructions = ""
    if compare:
        compare_instructions = textwrap.dedent("""
        Comparison mode: Compare FILE 1 and FILE 2 and include:
        - key similarities
        - key differences
        - likely operational impact
        - recommended actions to reconcile differences
        """)

    user_content = textwrap.dedent(f"""
    Context:
    {context}
    Data:
    {data_text}
    Instructions:
    {INTERPRET_INSTRUCTIONS}
    {compare_instructions}
    """)

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": user_content}
        ],
        "stream": False,
    }

    request_data = json.dumps(payload).encode("utf-8")
    request_url = urllib.request.Request(
        f"{normalize_host(host)}/api/chat",
        data=request_data,
        method="POST",
        headers={"Content-Type": "application/json"},
    )

    try:
        with urllib.request.urlopen(request_url, timeout=timeout_seconds) as response:
            return json.loads(response.read().decode("utf-8"))["message"]["content"]
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="ignore") if hasattr(e, "read") else ""
        raise RuntimeError(f"Interpretation request failed: HTTP {e.code} {e.reason}. Body: {body}") from e

## Run Sample Log Analysis
This runs the log interpretation helper on sample network events to show expected output format.

**Key Definitions:**
- `BGP adjacency`: Neighbor relationship required for route exchange.
- `Hold time expired`: BGP session dropped because keepalives were missed.
- `Link flap`: Interface repeatedly transitions up and down.
- `Correlation`: Matching multiple events over time to infer root cause.

**Purpose:**
Demonstrate real-world usage of the log interpretation function with practical examples. The commented sample shows BGP troubleshooting (hold timer expiration correlated with link flaps), while the active example performs a two-file comparison to identify removed servers between 2023 and 2026 configurations. This demonstrates how structured prompts can extract specific information (server names, details) in machine-readable JSON format, suitable for automated change tracking, compliance validation, or migration verification workflows.

In [34]:
# sample_data = """
# Feb 24 10:21:04 edge1 bgpd[1234]: %BGP-5-ADJCHANGE: neighbor 10.10.10.2 Down BGP Notification sent
# Feb 24 10:21:05 edge1 bgpd[1234]: %BGP-3-NOTIFICATION: received from neighbor 10.10.10.2 4/0 (hold time expired) 0 bytes
# Feb 24 10:21:08 edge1 kernel: eth0: Link is Down
# Feb 24 10:21:12 edge1 kernel: eth0: Link is Up - 10Gbps Full Duplex
# """

# print("Pasted-data mode:")
# print(
#     interpret_data(
#         pasted_data=sample_data,
#         context="Customer reports intermittent route loss to upstream; this is a PE in MPLS core.",
#     )
# )

file_a = "../assets/documents/voip_config_2023.txt"
file_b = "../assets/documents/voip_config_2026.txt"

# print("\nTwo-file analysis mode:")
# print(
#     interpret_data(
#         file_paths=[file_a, file_b],
#         context="Analyze both files and summarize important network patterns and risks.",
#         max_mb=0.5,
#         max_lines=1200,
#     )
# )

compare_context_prompt = """
Compare FILE 1 (2023) to FILE 2 (2026).
Goal: Identify changes and trends in the network configuration over time.
Output requirements:
1) Highlight significant differences between the two files.
2) Assess potential risks or improvements based on the changes.
3) Provide recommendations for maintaining or improving network reliability and performance based on the observed trends.
4) If applicable, identify any removed configurations that could lead to issues and any new configurations that could mitigate risks.
5) Keep the analysis focused on actionable insights for network operations teams.

"""

print("\nTwo-file compare mode:")
print(
    interpret_data(
        file_paths=[file_a, file_b],
        compare=True,
        context=compare_context_prompt,
        max_mb=0.5,
        max_lines=1200,
    )
)



Two-file compare mode:
**Summary of Key Insights**

* The VoIP configuration has undergone significant changes between 2023 and 2026, with updates to SIP trunks, dial plans, voicemail settings, call quality parameters, SBC endpoints, and security policies.
* The changes indicate a shift towards improved call quality, increased security, and enhanced features such as Teams integration and cloud-based voicemail.
* The updates also suggest a move towards more stringent security policies, including the enforcement of TLS 1.3 and SRTP for all calls.

**Key Details Supporting the Summary**

* **SIP Trunk Configuration:**
	+ New SIP trunks added: `sip_trunk_teams_integration` and `sip_trunk_backup_new`.
	+ Changes to existing SIP trunks: increased bandwidth for `sip_trunk_primary` and updated encryption for `sip_trunk_voicemailgw`.
* **Dial Plan:**
	+ New dial plan patterns added: `pattern_teams_federated` and `pattern_555[0-9]{4}` removed.
	+ Changes to existing dial plan patterns: updated 

## Analyze Images with the Model
This section adds an image-aware helper for extracting structured observations from screenshots or diagrams.

**Key Definitions:**
- `Multimodal`: Model can process both text and image inputs.
- `Base64`: Binary-to-text encoding used to send image bytes in JSON.
- `option_profile`: Named preset for latency/quality trade-offs.
- `retries`: Automatic reattempts for transient failures.
- `timeout_seconds`: Per-request wait limit before aborting.

**Purpose:**
Enable vision capabilities for analyzing screenshots, dashboards, network diagrams, or monitoring displays. The function handles image encoding, provides configurable performance profiles (fast/balanced/quality) for different use cases, and implements retry logic with exponential backoff for production reliability. This is essential for operational scenarios like analyzing monitoring dashboards, reading alarm panels, extracting data from screenshots, or interpreting topology diagrams - enabling AI assistance without manual transcription of visual information.

In [35]:
import base64, pathlib, json, urllib.request, urllib.error, socket, time

OPTION_PROFILES = {
    # Lower-latency profile: shorter generation and lower temperature.
    "fast": {"num_predict": 96, "temperature": 0.0, "top_p": 0.9},
    # # Balanced default profile for moderate latency and quality.
    # "balanced": {"num_predict": 192, "temperature": 0.1, "top_p": 0.95},
    # # Higher-quality profile for richer responses at higher latency.
    # "quality": {"num_predict": 384, "temperature": 0.2, "top_p": 0.95},
}

def normalize_host(host: str) -> str:
    return host.rstrip("/")

def chat_with_image(
    image_path: str,
    user_text: str,
    host=HOST,
    model=MODEL_NAME,
    timeout_seconds: int = 420,
    retries: int = 2,
    retry_backoff_seconds: int = 5,
    option_profile: str | None = "fast",
    options: dict | None = None,
) -> str:
    path = pathlib.Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    image_bytes = path.read_bytes()
    image_b64 = base64.b64encode(image_bytes).decode("utf-8")

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": user_text,
                "images": [image_b64],
            }
        ],
        "stream": False,
    }

    selected_options = {}
    if option_profile:
        if option_profile not in OPTION_PROFILES:
            supported = ", ".join(sorted(OPTION_PROFILES.keys()))
            raise ValueError(f"Unknown option_profile '{option_profile}'. Supported: {supported}")
        selected_options.update(OPTION_PROFILES[option_profile])
    if options:
        selected_options.update(options)
    if selected_options:
        payload["options"] = selected_options

    host = normalize_host(host)
    request_data = json.dumps(payload).encode("utf-8")
    request_url = urllib.request.Request(
        f"{host}/api/chat",
        data=request_data,
        method="POST",
        headers={"Content-Type": "application/json"},
    )

    last_error = None
    for attempt in range(retries + 1):
        try:
            with urllib.request.urlopen(request_url, timeout=timeout_seconds) as response:
                body = json.loads(response.read().decode("utf-8"))
                return body["message"]["content"]
        except TimeoutError as e:
            last_error = e
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf-8", errors="ignore") if hasattr(e, "read") else ""
            raise RuntimeError(f"Ollama chat_with_image failed: HTTP {e.code} {e.reason}. Body: {detail}") from e
        except urllib.error.URLError as e:
            last_error = e

        if attempt < retries:
            time.sleep(retry_backoff_seconds * (attempt + 1))

    raise TimeoutError(
        f"Timed out after {retries + 1} attempts calling {host}/api/chat with timeout={timeout_seconds}s"
    ) from last_error

## Run Image Analysis Example
This executes an example vision prompt and returns structured output for operational reporting workflows.

**Key Definitions:**
- `JSON-only prompt`: Instruction that constrains response shape for machine-readability.
- `BUS-ID`: PCI bus identifier used to locate a GPU device.
- `Inference latency`: Total time from request send to full response.
- `Profile override`: Per-call tuning that supersedes default generation settings.

**Purpose:**
Demonstrate practical vision model usage by extracting GPU metrics (temperature, power usage, bus ID) from a screenshot in structured JSON format. This shows how to combine vision capabilities with structured output requirements for automation workflows. The fast profile ensures low latency for real-time monitoring scenarios. This pattern is applicable to extracting metrics from any dashboard, monitoring interface, or equipment display - enabling automated data collection from systems that don't provide APIs.

In [36]:
print(
    chat_with_image(
        "../assets/llamaDemoPic.webp",
        "Can you give me a JSON response of the GPU number, its temperature, power usage along with BUS-ID?",
        option_profile="fast",
        timeout_seconds=420,
        retries=1,
    )
)

Here is the JSON response:

```json
[
  {
    "GPU": 0,
    "Temperature": 73,
    "Power": 500,
    "BUS-ID": "00000000:06:00.0"
  },
  {
    "GPU": 1,
    "Temperature": 71,
    "Power": 500,
    "BUS-ID": "00000000:26:00.0"
  },
  {
    "


## Define Few-Shot Examples
This section prepares a library of labeled examples that demonstrate the expected interpretation format for log data.

**Key Definitions:**
- `Few-shot learning`: Technique where a model learns to perform a task from a small set of examples shown in the prompt.
- `Example structure`: Each entry includes input (log snippet) and output (structured interpretation).
- `Output format`: Consistent format with summary, key details, root causes, next steps, and verification.
- `Domain diversity`: Examples cover BGP networking, interface errors, and VoIP scenarios.

**Purpose:**
Establish a reusable template library that teaches the model the exact format expected for log interpretation. By providing multiple worked examples with correct output structure before asking for analysis, the model learns to replicate that format reliably. This "in-context learning" approach dramatically improves consistency and reduces malformed responses, eliminating the need for explicit JSON schema specifications.

In [37]:
FEW_SHOT_EXAMPLE = """
Example 1:

Input:
[2024-06-01 12:00:00] bgpd: neighbor 10.10.10.2 Down
[2024-06-01 12:01:00] kernel: eth0: Link is Down
[2024-06-01 12:02:00] kernel: eth0: Link is Up
[2024-06-01 12:08:00] bgpd: neighbor 10.10.10.2 Up

Output:
1) Summary:
- BGP session dropped due to link flaps on eth0.

2) Key details:
- At 12:00, BGP neighbor 10.10.10.2 went down following eth0 link down.
- Likely root cause:
- Physical link instability or flapping on eth0.

3) Recommended next steps:
- Check counters for eth0 to confirm link flaps.
- Check physical connections and SFPs for eth0.

4) Verification steps:
- Monitor eth0 for link stability.
- Monitor BGP session for stability after addressing physical layer.

Example 2:
Input:
[2024-06-01 12:00:00] bgpd: neighbor 10.10.10.2 Down
[2024-06-01 12:01:00] interface Gi0/1: Input errors 100, CRC 50
[2024-06-01 12:02:00] interface Gi0/1: Input errors 150, CRC 75
[2024-06-01 12:08:00] bgpd: neighbor 10.10.10.2 Up

Output:
1) Summary:
- BGP session dropped due to increasing input errors and CRC errors on interface Gi0/1.

2) Key details:
- At 12:00, BGP neighbor 10.10.10.2 went down following increasing input errors and CRC errors on interface Gi0/1.
- Likely root cause:
- Interface Gi0/1 experiencing high input errors and CRC errors.

3) Recommended next steps:
- Check counters for interface Gi0/1 to confirm error rates.
- Check physical connections and SFPs for interface Gi0/1.

4) Verification steps:
- Monitor interface Gi0/1 for error rates.
- Monitor BGP session for stability after addressing interface issues.

Example 3:
Input:
SIP 408 Request Timeout for call from 1234567890 to 0987654321
RTP stream from 1234567890 to 0987654321 shows high packet loss and jitter

Output:
1) Summary:
- VoIP call from 1234567890 to 0987654321 is experiencing quality issues and timing out due to high packet loss and jitter in the RTP stream.

2) Key details:
- SIP 408 Request Timeout indicates the call setup failed.
- RTP stream shows high packet loss and jitter, affecting call quality.

3) Recommended next steps:
- Investigate network conditions between the endpoints.
- Check for congestion or packet loss in the network path.

4) Verification steps:
- Monitor RTP stream for improvements after addressing network issues.
- Test call setup to ensure SIP requests are successful.
"""



## Implement Few-Shot Interpreter Function
This function wraps the chat interface to inject the example library and enforce JSON output format.

**Key Definitions:**
- `Few-shot prompt injection`: Embedding examples directly into the user prompt to guide model behavior.
- `Error handling`: Try-catch block to gracefully handle non-JSON responses from the model.
- `JSON fallback`: Returns raw response wrapped as `{"raw_response": ...}` if parsing fails.
- `Prompt template`: Dynamic formatting that inserts examples and user data into a structured query.

**Purpose:**
Create a production-ready wrapper around the basic chat function that adds few-shot learning capability. This function automatically inserts the example library into every request, enabling the model to produce consistently formatted JSON output without separate training or fine-tuning. The error handling ensures robustness when the model deviates from the expected format, preventing hard failures in automation pipelines.

In [38]:
def interpret_with_fewshot(pasted_data: str):
    prompt = f"""
Use the format from the examples below to interpret the following data. Focus on providing actionable insights in the output.

{FEW_SHOT_EXAMPLE}

Now analyze this data:

Input:
{pasted_data}

Output (respond with valid JSON only):
"""
    response = chat(prompt)
    content = response["message"]["content"]
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"raw_response": content}

## Validate Few-Shot Interpreter with Real Data
This runs a test case through the few-shot interpreter to verify the learned format is applied correctly.

**Key Definitions:**
- `Test data`: Real-world operational data (VoIP call timeout and RTP quality issues).
- `Model output`: JSON response containing structured interpretation.
- `Format validation`: Confirms the model followed the few-shot format despite seeing new data.
- `Round-trip verification`: Confirms the entire pipeline (prompt → model → parsing) works end-to-end.

**Purpose:**
Execute a practical validation that the few-shot learning approach works as intended on unseen data. This test case uses VoIP metrics similar to the examples but with different specific values, confirming that the model generalizes from the examples rather than memorizing them. Success here indicates the approach is production-ready for handling new operational logfiles and metrics in the established format.

In [39]:
test_data = """
SIP 308 Request Timeout for call from 1234567890 to 0987654321
RTP stream from 1234567890 to 0987654321 shows high packet loss and jitter
"""
print(json.dumps(interpret_with_fewshot(test_data), indent=2))

{
  "1) Summary": "VoIP call from 1234567890 to 0987654321 is experiencing quality issues and timing out due to high packet loss and jitter in the RTP stream.",
  "2) Key details": [
    "SIP 308 Request Timeout indicates the call setup failed.",
    "RTP stream shows high packet loss and jitter, affecting call quality."
  ],
  "3) Recommended next steps": [
    "Investigate network conditions between the endpoints.",
    "Check for congestion or packet loss in the network path."
  ],
  "4) Verification steps": [
    "Monitor RTP stream for improvements after addressing network issues.",
    "Test call setup to ensure SIP requests are successful."
  ]
}
